# TechJam Track 5 — full pipeline on ColabRuntime → Change runtime type → **T4 GPU** before running.Runs everything: data download (~50GB into Colab's disk), manifests, training (cnn + clip_linear), evaluation on WildFake test AND the official benchmark (DALL·E-Advanced + COCO val2017). Only small artifacts leave Colab.

In [ ]:
!git clone https://github.com/wheres-my-perry/techjam-2026-track5.git%cd techjam-2026-track5!pip -q install -r requirements-train.txtimport torchprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable GPU runtime!")

In [ ]:
# optional but recommended: persist results to your Google Drivefrom google.colab import drivedrive.mount('/content/drive')import osPERSIST = '/content/drive/MyDrive/techjam_outputs'os.makedirs(PERSIST, exist_ok=True)print('persisting to', PERSIST)

## 1. Data (~50GB download; 30–90 min depending on ModelScope bandwidth)

In [ ]:
!python scripts/get_wildfake.py --include 'label_csv_files/**' --include 'dataset_infos.json'

In [ ]:
!python scripts/get_wildfake.py --include 'Images/Real/coco.zip' --extract-filter val2017 --delete-zips

In [ ]:
!python scripts/get_wildfake.py --include 'Images/Real/imagenet.zip' --include 'Images/Real/ffhq.zip' --include 'Images/Real/church.zip' --include 'Images/Real/celebahq.zip' --include 'Images/Real/afhq.zip' --delete-zips

In [ ]:
!python scripts/get_wildfake.py --include 'Images/Diffusion_based/DDIM.zip' --include 'Images/Diffusion_based/DDPM.zip' --delete-zips

In [ ]:
# benchmark fakes: 23.8GB zip, keeps only the 8,843 dalle3 images then deletes the zip!python scripts/get_wildfake.py --include 'Images/Diffusion_based/DALLE.zip' --extract-filter dalle3 --delete-zips

## 2. Manifests (DDPM held out of training = extra unseen-generator test)

In [ ]:
!python scripts/get_wildfake.py --manifest --holdout-generator ddpm!python scripts/get_wildfake.py --official-val

## 3. Train both approaches

In [ ]:
!python -m src.approaches.cnn.train --train data/manifests/wildfake_train.csv --val data/manifests/wildfake_val.csv --epochs 5 --width 64 --augment --crop 224 --batch 64 --out outputs/cnn/wf_aug_w64.pt

In [ ]:
!python -m src.approaches.clip_linear.train --train data/manifests/wildfake_train.csv --val data/manifests/wildfake_val.csv --backbone ViT-L-14 --augment-views 2 --out outputs/clip_linear/wf_l14_aug.pt

## 4. Evaluate: robustness grid + per-generator (incl. held-out ddpm) + OFFICIAL benchmark

In [ ]:
!python -m src.evaluate --manifest data/manifests/wildfake_test.csv --model cnn:outputs/cnn/wf_aug_w64.pt --out outputs/cnn/eval_wf_test --limit 4000

In [ ]:
!python -m src.evaluate --manifest data/manifests/official_val.csv --model cnn:outputs/cnn/wf_aug_w64.pt --out outputs/cnn/eval_official --limit 4000

In [ ]:
!python -m src.evaluate --manifest data/manifests/wildfake_test.csv --model clip_linear:outputs/clip_linear/wf_l14_aug.pt --out outputs/clip_linear/eval_wf_test --limit 4000

In [ ]:
!python -m src.evaluate --manifest data/manifests/official_val.csv --model clip_linear:outputs/clip_linear/wf_l14_aug.pt --out outputs/clip_linear/eval_official --limit 4000

## 5. Bring results home (weights + tables; embedding cache excluded)

In [ ]:
import shutil, osshutil.copytree('outputs', os.path.join(PERSIST, 'outputs'), dirs_exist_ok=True,                ignore=shutil.ignore_patterns('cache'))!zip -qr results_bundle.zip outputs -x 'outputs/clip_linear/cache/*'from google.colab import filesfiles.download('results_bundle.zip')

Back on the Mac: unzip `results_bundle.zip` into the repo root (merges into `outputs/`), then commit + push.